# Лабораторная работа №9
## Методы обучения без учителя: PCA, t-SNE, кластеризация

Методичка: [LAB_TMO_CLUSTER](https://github.com/ugapanyuk/courses_current/wiki/LAB_TMO_CLUSTER).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
%matplotlib inline

RANDOM_STATE = 42

In [ ]:
wine = load_wine(as_frame=True)
df = wine.frame.copy()
target = df["target"].astype(int)

feature_subset = ["alcohol", "malic_acid", "flavanoids", "color_intensity", "hue", "proline"]
D1 = df[feature_subset].copy()

scaler = StandardScaler()
D1s = scaler.fit_transform(D1)
D1s = pd.DataFrame(D1s, columns=feature_subset, index=D1.index)
D1s.head()

In [ ]:
D2 = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(D1s)
D2 = pd.DataFrame(D2, columns=["pc1", "pc2"], index=D1s.index)

D3 = TSNE(n_components=2, init="pca", learning_rate="auto", perplexity=30, random_state=RANDOM_STATE).fit_transform(D1s)
D3 = pd.DataFrame(D3, columns=["tsne1", "tsne2"], index=D1s.index)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(x=D2["pc1"], y=D2["pc2"], hue=target, palette="tab10", ax=axes[0], s=40, alpha=0.85)
axes[0].set_title("PCA (D2)")
sns.scatterplot(x=D3["tsne1"], y=D3["tsne2"], hue=target, palette="tab10", ax=axes[1], s=40, alpha=0.85)
axes[1].set_title("t-SNE (D3)")
plt.tight_layout()
plt.show()

In [ ]:
datasets = {
    "D1": D1s.values,
    "D2": D2.values,
    "D3": D3.values,
}

clusterers = {
    "KMeans": KMeans(n_clusters=3, n_init=20, random_state=RANDOM_STATE),
    "Agglomerative": AgglomerativeClustering(n_clusters=3, linkage="ward"),
    "DBSCAN": DBSCAN(eps=0.8, min_samples=6),
}

def metrics(X, labels):
    labels = np.asarray(labels)
    mask = labels != -1
    X2 = X[mask]
    lab2 = labels[mask]
    ncl = len(set(lab2))
    if ncl < 2:
        return ncl, np.nan, np.nan, np.nan, float((labels == -1).mean())
    return (
        ncl,
        float(silhouette_score(X2, lab2)),
        float(calinski_harabasz_score(X2, lab2)),
        float(davies_bouldin_score(X2, lab2)),
        float((labels == -1).mean()),
    )

rows = []
labels_store = {}
for dname, X in datasets.items():
    for cname, cl in clusterers.items():
        lab = cl.fit_predict(X)
        labels_store[(dname, cname)] = lab
        ncl, sil, ch, db, noise = metrics(X, lab)
        rows.append({"dataset": dname, "method": cname, "n_clusters": ncl, "silhouette": sil, "CH": ch, "DB": db, "noise_frac": noise})

res = pd.DataFrame(rows)
display(res.round(4))